In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

In [2]:
import os

# Oyaage Kaggle username saha key eka methanata danna
os.environ['KAGGLE_USERNAME'] = "devindithathsara "
os.environ['KAGGLE_KEY'] = "29219555"

print("⏳ Downloading dataset from Kaggle...")
!pip install -q kaggle
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip -d dataset/

import pandas as pd
from sklearn.preprocessing import StandardScaler

print("⏳ Loading and preprocessing dataset...")
df = pd.read_csv('dataset/creditcard.csv')

# scalling time and amount (anith ewa kalinma PCA walin scale wela thiyenne)
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['scaled_time'] = scaler.fit_transform(df['Time'].values.reshape(-1,1))

#remove old features
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# X (features) saha y (labels) widihata wen kirima
X = df.drop('Class', axis=1).values
y = df['Class'].values

print(f"✅ Dataset ready! Total transactions: {len(df)}")

⏳ Downloading dataset from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 202MB/s]

Archive:  creditcardfraud.zip
  inflating: dataset/creditcard.csv  
⏳ Loading and preprocessing dataset...
✅ Dataset ready! Total transactions: 284807


In [3]:
from sklearn.model_selection import StratifiedKFold, train_test_split

NUM_CLIENTS = 12

skf = StratifiedKFold(n_splits=NUM_CLIENTS, shuffle=True, random_state=42)

clients_final_data = {}
client_id = 0

print("⏳ Distributing data to 12 banks...")

⏳ Distributing data to 12 banks...


In [4]:
for train_idx, client_idx in skf.split(X, y):
    X_client = X[client_idx]
    y_client = y[client_idx]

    # for each bank; local data train (80%) , test (20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X_client, y_client, test_size=0.2, random_state=42, stratify=y_client
    )

    # RAM eka ithuru karaganna float32 widihata save kirima
    clients_final_data[client_id] = {
        "X_train": np.array(X_train, dtype='float32'),
        "y_train": np.array(y_train, dtype='float32'),
        "X_test": np.array(X_test, dtype='float32'),
        "y_test": np.array(y_test, dtype='float32')
    }
    client_id += 1

print("✅ Data distributed successfully!\n")

✅ Data distributed successfully!



In [5]:
# check data(bankuwakata bedila tyn data gaana)
for i in range(NUM_CLIENTS):
    total = len(clients_final_data[i]['y_train']) + len(clients_final_data[i]['y_test'])
    frauds = int(sum(clients_final_data[i]['y_train']) + sum(clients_final_data[i]['y_test']))
    print(f"🏦 Bank {i+1}: Total Transactions = {total} | Frauds = {frauds}")

🏦 Bank 1: Total Transactions = 23734 | Frauds = 41
🏦 Bank 2: Total Transactions = 23734 | Frauds = 41
🏦 Bank 3: Total Transactions = 23734 | Frauds = 41
🏦 Bank 4: Total Transactions = 23734 | Frauds = 41
🏦 Bank 5: Total Transactions = 23734 | Frauds = 41
🏦 Bank 6: Total Transactions = 23734 | Frauds = 41
🏦 Bank 7: Total Transactions = 23734 | Frauds = 41
🏦 Bank 8: Total Transactions = 23734 | Frauds = 41
🏦 Bank 9: Total Transactions = 23734 | Frauds = 41
🏦 Bank 10: Total Transactions = 23734 | Frauds = 41
🏦 Bank 11: Total Transactions = 23734 | Frauds = 41
🏦 Bank 12: Total Transactions = 23733 | Frauds = 41


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

# create Model function
# මෙහි විශේෂත්වය වන්නේ BatchNormalization layers ඇතුළත් කිරීමයි (FedBN සඳහා)
def create_model():
    model = models.Sequential([
        layers.Input(shape=(30,)), # Features 30 (V1-V28 + Scaled Time + Scaled Amount)
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(), # FedBN වලදී මේ layer එක share කරන්නේ නැහැ
        layers.Dropout(0.2),

        layers.Dense(16, activation='relu'),
        layers.BatchNormalization(), # FedBN වලදී මේ layer එක share කරන්නේ නැහැ

        layers.Dense(1, activation='sigmoid') # Binary Classification (Fraud or Not)
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [7]:
# create Global Models සහ Local Client Models
print("⏳ Creating models for 12 banks...")

# FedAvg
global_fedavg_model = create_model()

# FedBN
global_fedbn_model = create_model()

# බැංකු 12 සඳහා Local Models (FedBN වලදී මේවා මතකයේ තබා ගත යුතුයි)
client_models = [create_model() for _ in range(NUM_CLIENTS)]

⏳ Creating models for 12 banks...


In [8]:
# give same  Weights
initial_weights = global_fedbn_model.get_weights()
for model in client_models:
    model.set_weights(initial_weights)

print("✅ Global and Local Models are created and initialized.")

✅ Global and Local Models are created and initialized.


In [10]:
import gc

# Training Parameters
NUM_ROUNDS = 20
LOCAL_EPOCHS = 1
BATCH_SIZE = 32

fedbn_history = []

print("🚀 Starting Corrected Federated Training Loop (FedBN)...")

for round_num in range(NUM_ROUNDS):
    print(f"--- Global Round {round_num+1}/{NUM_ROUNDS} ---")

    # 1. සියලුම බැංකු වල weights තාවකාලිකව තබා ගැනීමට ලැයිස්තුවක්
    all_client_layer_weights = []

    # --- Local Training for each Bank ---
    for client_id in range(NUM_CLIENTS):
        X_train = clients_final_data[client_id]["X_train"]
        y_train = clients_final_data[client_id]["y_train"]

        local_model = client_models[client_id]

        # දත්ත ප්‍රමාණය අනුව fit කිරීම
        local_model.fit(X_train, y_train, epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)

        # හැම layer එකකම weights වෙන වෙනම list එකකට ගන්න
        weights_per_layer = [layer.get_weights() for layer in local_model.layers]
        all_client_layer_weights.append(weights_per_layer)

    # --- FedBN Aggregation (Layer by Layer) ---
    for i, layer in enumerate(global_fedbn_model.layers):
        # Layer එකේ weights තියෙනවා නම් සහ එය BatchNormalization නොවේ නම් පමණක් එකතු කරන්න
        if len(layer.get_weights()) > 0 and not isinstance(layer, tf.keras.layers.BatchNormalization):
            # සියලුම clients ගේ i-වැනි layer එකේ weights ලබා ගැනීම
            layer_weights_all_clients = [cw[i] for cw in all_client_layer_weights]

            # Kernel සහ Bias වෙන වෙනම average කිරීම
            avg_layer_weights = []
            for weight_idx in range(len(layer.get_weights())):
                avg_w = np.mean([w[weight_idx] for w in layer_weights_all_clients], axis=0)
                avg_layer_weights.append(avg_w)

            # Global model එකේ i-වැනි layer එක update කිරීම
            layer.set_weights(avg_layer_weights)

    # --- Syncing Non-BN weights back to clients ---
    for client_id in range(NUM_CLIENTS):
        for i, layer in enumerate(global_fedbn_model.layers):
            # BN layers හැර අනිත් ඒවා global model එකෙන් local models වලට copy කිරීම
            if len(layer.get_weights()) > 0 and not isinstance(layer, tf.keras.layers.BatchNormalization):
                client_models[client_id].layers[i].set_weights(layer.get_weights())

    # --- Evaluation ---
    current_bn_accs = []
    for client_id in range(NUM_CLIENTS):
        _, acc = client_models[client_id].evaluate(clients_final_data[client_id]["X_test"],
                                                   clients_final_data[client_id]["y_test"], verbose=0)
        current_bn_accs.append(acc)

    avg_bn_acc = np.mean(current_bn_accs)
    fedbn_history.append(avg_bn_acc)

    print(f"✅ Round {round_num+1} Done. Avg FedBN Accuracy: {avg_bn_acc:.4f}")
    gc.collect()

print("\n🎯 Training Complete with FedBN!")

🚀 Starting Corrected Federated Training Loop (FedBN)...
--- Global Round 1/20 ---
✅ Round 1 Done. Avg FedBN Accuracy: 0.9992
--- Global Round 2/20 ---
✅ Round 2 Done. Avg FedBN Accuracy: 0.9992
--- Global Round 3/20 ---
✅ Round 3 Done. Avg FedBN Accuracy: 0.9993
--- Global Round 4/20 ---
✅ Round 4 Done. Avg FedBN Accuracy: 0.9993
--- Global Round 5/20 ---
✅ Round 5 Done. Avg FedBN Accuracy: 0.9993
--- Global Round 6/20 ---
✅ Round 6 Done. Avg FedBN Accuracy: 0.9992
--- Global Round 7/20 ---
✅ Round 7 Done. Avg FedBN Accuracy: 0.9993
--- Global Round 8/20 ---
✅ Round 8 Done. Avg FedBN Accuracy: 0.9993
--- Global Round 9/20 ---
✅ Round 9 Done. Avg FedBN Accuracy: 0.9992
--- Global Round 10/20 ---
✅ Round 10 Done. Avg FedBN Accuracy: 0.9992
--- Global Round 11/20 ---
✅ Round 11 Done. Avg FedBN Accuracy: 0.9993
--- Global Round 12/20 ---
✅ Round 12 Done. Avg FedBN Accuracy: 0.9994
--- Global Round 13/20 ---
✅ Round 13 Done. Avg FedBN Accuracy: 0.9992
--- Global Round 14/20 ---
✅ Round 14 D

In [11]:
local_final_acc = []

print("📊 Starting Baseline Training (Individual Banks - No FL)...")

for client_id in range(NUM_CLIENTS):
    # අලුත්ම model එකක් එක් එක් බැංකුවට නිර්මාණය කිරීම
    individual_model = create_model()

    X_train = clients_final_data[client_id]["X_train"]
    y_train = clients_final_data[client_id]["y_test"] # මේක test label නෙවෙයි y_train වෙන්න ඕනේ

    # ඇත්තටම මෙතැනදී වැරදීමකින් y_test දාන්න එපා, y_train ම පාවිච්චි කරන්න
    y_train = clients_final_data[client_id]["y_train"]

    # එක බැංකුවක් තනියම train කරනවා (FL නැතිව)
    # අපි FL වලදී round 20 ක් run කළ නිසා මෙතනත් epochs 20 ක් දාමු (Fair comparison එකක් සඳහා)
    individual_model.fit(X_train, y_train, epochs=20, batch_size=BATCH_SIZE, verbose=0)

    # Evaluate accuracy
    _, acc = individual_model.evaluate(clients_final_data[client_id]["X_test"],
                                      clients_final_data[client_id]["y_test"], verbose=0)
    local_final_acc.append(acc)
    print(f"🏦 Bank {client_id+1} Individual Accuracy: {acc:.4f}")

print("\n✅ Baseline Training Complete!")

📊 Starting Baseline Training (Individual Banks - No FL)...
🏦 Bank 1 Individual Accuracy: 0.9989
🏦 Bank 2 Individual Accuracy: 0.9994
🏦 Bank 3 Individual Accuracy: 0.9989
🏦 Bank 4 Individual Accuracy: 0.9989
🏦 Bank 5 Individual Accuracy: 0.9989
🏦 Bank 6 Individual Accuracy: 0.9998
🏦 Bank 7 Individual Accuracy: 0.9996
🏦 Bank 8 Individual Accuracy: 0.9996
🏦 Bank 9 Individual Accuracy: 0.9992
🏦 Bank 10 Individual Accuracy: 0.9987
🏦 Bank 11 Individual Accuracy: 0.9996
🏦 Bank 12 Individual Accuracy: 0.9994

✅ Baseline Training Complete!
